# 01 – Data audit 

Goal: find every data problem in the four raw files, count how many rows it affects,
and decide whether to **fix, flag or drop** it, with a reason.
Findings are collected in an issue log and summarized in `findings.md`.

Raw files are never edited; every change is made in code so it can be reproduced
and later applied inside the cleaning transform.

In [1]:
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")
FILES = {
    "applications": "applications.csv",
    "bureau": "credit_bureau.csv",
    "servicing": "loan_servicing.csv",
    "holdout": "holdout_applications.csv",
}

def load_raw(raw_dir=RAW):
    """Read each raw file; force id to text so ids like '0123456789' keep their leading zero."""
    return {name: pd.read_csv(raw_dir / fname, dtype={"id": str}, low_memory=False)
            for name, fname in FILES.items()}

data = load_raw()
apps, bureau, serv, hold = (data[k] for k in ["applications", "bureau", "servicing", "holdout"])

### Issue log
Every confirmed problem is recorded with its file, column, number of affected rows,
the action taken (fix / flag / drop) and the reason. At the end of the audit this
becomes the issue table in `findings.md`.

In [2]:
ISSUES = []

def log_issue(file, column, issue, rows, action, reason):
    """Record one data problem: where, what, how many rows, and the decision."""
    ISSUES.append({"file": file, "column": column, "issue": issue,
                   "rows": int(rows), "action": action, "reason": reason})

### Duplicates in each file
Two kinds of duplicates are counted:
- **Exact duplicates:** a whole row repeated.
- **Same id, different values:** the same loan recorded more than once with
  conflicting details, which is harder to resolve.

Duplicates would make a loan count twice in training and in the default rate.

In [3]:
def duplicate_report(df):
    """Two kinds of duplicates: identical copies, and same id with different values."""
    exact = df.duplicated().sum()                    # whole row repeated
    conflicting = df.drop_duplicates()["id"].duplicated().sum()  # same id, values differ
    return {"rows": len(df), "unique_ids": df["id"].nunique(),
            "exact_duplicates": int(exact), "same_id_different_values": int(conflicting)}

for name, df in data.items():
    print(name, duplicate_report(df))

applications {'rows': 30600, 'unique_ids': 30000, 'exact_duplicates': 0, 'same_id_different_values': 600}
bureau {'rows': 30000, 'unique_ids': 30000, 'exact_duplicates': 0, 'same_id_different_values': 0}
servicing {'rows': 30000, 'unique_ids': 30000, 'exact_duplicates': 0, 'same_id_different_values': 0}
holdout {'rows': 5000, 'unique_ids': 5000, 'exact_duplicates': 0, 'same_id_different_values': 0}


### What differs between conflicting copies?
For a few repeated ids, only the columns where the copies disagree are shown.
Knowing *what* differs tells us whether the copies are the same loan with
formatting differences, or genuinely different records.

In [4]:
def show_conflicts(df, n=3):
    """For a few ids that repeat with different values, show only the columns that differ."""
    deduped = df.drop_duplicates()
    ids = deduped.loc[deduped["id"].duplicated(), "id"].unique()[:n]
    for loan_id in ids:
        rows = deduped[deduped["id"] == loan_id]
        diff_cols = rows.columns[rows.nunique(dropna=False) > 1].tolist()
        print(f"\nid {loan_id} differs in: {diff_cols}")
        display(rows[["id"] + diff_cols])

show_conflicts(apps)


id 8acbab3c06 differs in: ['emp_title', 'dti']


,id,emp_title,dti
108,8acbab3c06,Hillstone,15.53
1199,8acbab3c06,HILLSTONE,15.54



id ddefd45498 differs in: ['emp_title', 'dti']


,id,emp_title,dti
429,ddefd45498,Safety Manager,15.36
1214,ddefd45498,SAFETY MANAGER,15.37



id 01edeb5c98 differs in: ['emp_title', 'dti']


,id,emp_title,dti
1083,01edeb5c98,Legal Negotiator,14.49
1506,01edeb5c98,LEGAL NEGOTIATOR,14.50


### Which columns differ, across all repeated ids?
Before deciding, we check that the pattern seen in the examples holds for every
repeated id. If an important field such as `defaulted` or `loan_amnt` ever
differed, the copies could not simply be merged.

In [7]:
def conflict_summary(df):
    """For every id that repeats with different values, count which columns differ."""
    deduped = df.drop_duplicates()
    dup = deduped[deduped["id"].duplicated(keep=False)]   # all copies of repeated ids
    counts = {}
    for loan_id, rows in dup.groupby("id"):
        for col in rows.columns[rows.nunique(dropna=False) > 1]:
            counts[col] = counts.get(col, 0) + 1
    print("Repeated ids:", dup["id"].nunique())
    return pd.Series(counts, dtype=int).sort_values(ascending=False)

conflict_summary(apps)

Repeated ids: 600


dti          600
emp_title    528
dtype: int64

In [8]:
dup = apps.drop_duplicates()
dup = dup[dup["id"].duplicated(keep=False)]
g = dup.groupby("id")

case_only = g["emp_title"].agg(lambda s: s.str.lower().nunique(dropna=False) == 1)
print("emp_title differs only by case:", case_only.all())
print("Largest dti difference:", (g["dti"].max() - g["dti"].min()).max())

emp_title differs only by case: True
Largest dti difference: 0.010000000000005116


### Decision: drop duplicate ids (keep first)
600 ids appear twice in `applications.csv`. The copies differ only in `emp_title`
capitalization (528 pairs) and `dti` by at most 0.01 (600 pairs); the label and
amounts never differ. They are the same loan entered twice, so the second copy is
dropped. Keeping both would double-count these loans.
This leaves 30,000 applications, matching the bureau and servicing files.

In [9]:
before = len(apps)
apps = apps.drop_duplicates(subset="id", keep="first").reset_index(drop=True)
print(f"applications: {before} → {len(apps)} rows")

log_issue(
    file="applications", column="id",
    issue="600 ids appear twice; copies differ only in emp_title capitalization (528) and dti by ≤0.01 (600)",
    rows=600,
    action="drop duplicate (keep first)",
    reason="Same loan entered twice; keeping both would double-count it. "
           "Differences are formatting/rounding, and the label and amounts never differ",
)

applications: 30600 → 30000 rows


### Do the three files cover the same loans?
Compares the sets of ids between applications, bureau and servicing. A loan
missing from one file would lose its bureau data or its servicing check when joined.

In [5]:
def coverage(df_a, df_b, name_a, name_b):
    """Ids present in one file but not the other."""
    a, b = set(df_a["id"]), set(df_b["id"])
    print(f"{name_a} ids missing from {name_b}: {len(a - b)}")
    print(f"{name_b} ids missing from {name_a}: {len(b - a)}")

coverage(apps, bureau, "applications", "bureau")
coverage(apps, serv, "applications", "servicing")
coverage(bureau, serv, "bureau", "servicing")

applications ids missing from bureau: 0
bureau ids missing from applications: 0
applications ids missing from servicing: 0
servicing ids missing from applications: 0
bureau ids missing from servicing: 0
servicing ids missing from bureau: 0


### Does the holdout overlap with training?
Holdout loans are from 2016 and should not appear in the training files.
Any overlap would put the same loan in both training and final scoring.

In [6]:
overlap = set(hold["id"]) & set(apps["id"])
print("Holdout ids also in applications:", len(overlap))

Holdout ids also in applications: 0


## A1.2 Formats

### Date formats
Each date value is reduced to its "shape" (letters → `A`, digits → `9`), so
`Dec-2013` becomes `AAA-9999` and `2013-12-01` becomes `9999-99-99`. More than one
shape in a column means mixed date formats, which can be parsed incorrectly or
silently, for example by swapping day and month.

In [10]:
def pattern_counts(s):
    """Reduce each value to its 'shape': letters become A, digits become 9."""
    shapes = (s.dropna().astype(str)
              .str.replace(r"[A-Za-z]", "A", regex=True)
              .str.replace(r"\d", "9", regex=True))
    return shapes.value_counts()

DATE_COLS = {
    "applications": ["issue_d"],
    "bureau": ["earliest_cr_line"],
    "servicing": ["last_pymnt_d", "last_credit_pull_d"],
    "holdout": ["issue_d", "earliest_cr_line"],
}

for name, cols in DATE_COLS.items():
    for col in cols:
        print(f"\n--- {name}.{col} ---")
        print(pattern_counts(data[name][col] if name != "applications" else apps[col]))


--- applications.issue_d ---
issue_d
AAA-9999      28800
9999-99-99      615
99/9999         585
Name: count, dtype: int64

--- bureau.earliest_cr_line ---
earliest_cr_line
AAA-9999      28800
9999-99-99      607
99/9999         593
Name: count, dtype: int64

--- servicing.last_pymnt_d ---
last_pymnt_d
AAA-9999    29979
Name: count, dtype: int64

--- servicing.last_credit_pull_d ---
last_credit_pull_d
AAA-9999    29999
Name: count, dtype: int64

--- holdout.issue_d ---
issue_d
AAA-9999    5000
Name: count, dtype: int64

--- holdout.earliest_cr_line ---
earliest_cr_line
AAA-9999    5000
Name: count, dtype: int64


### Case and spacing variants in categories
Counts distinct values before and after removing surrounding spaces and lowercasing.
If normalizing reduces the count, the same category is written in more than one way
(e.g. `RENT` vs `rent `), which would split one group into several.
The holdout is checked too, because it may contain variants that training does not.

In [11]:
TEXT_COLS = ["term", "grade", "sub_grade", "purpose", "application_type",
             "emp_length", "home_ownership", "verification_status", "addr_state"]

def variant_check(df, cols):
    """Distinct values before vs after normalizing case and spaces."""
    rows = []
    for c in cols:
        raw = df[c].dropna().astype(str)
        rows.append({"column": c, "distinct_raw": raw.nunique(),
                     "distinct_normalized": raw.str.strip().str.lower().nunique()})
    out = pd.DataFrame(rows)
    out["has_variants"] = out["distinct_raw"] > out["distinct_normalized"]
    return out

print("applications"); display(variant_check(apps, TEXT_COLS))
print("holdout"); display(variant_check(hold, TEXT_COLS))

applications


,column,distinct_raw,distinct_normalized,has_variants
0,term,2,2,False
1,grade,7,7,False
2,sub_grade,35,35,False
3,purpose,13,13,False
4,application_type,2,2,False
5,emp_length,11,11,False
6,home_ownership,3,3,False
7,verification_status,3,3,False
8,addr_state,154,61,True


holdout


,column,distinct_raw,distinct_normalized,has_variants
0,term,2,2,False
1,grade,7,7,False
2,sub_grade,35,35,False
3,purpose,12,12,False
4,application_type,2,2,False
5,emp_length,11,11,False
6,home_ownership,3,3,False
7,verification_status,3,3,False
8,addr_state,50,50,False


### Inspect the variants
For each column flagged above, lists every raw spelling grouped under its
normalized value, with counts, to decide how to standardize them.

In [12]:
def show_variants(df, col):
    """Group raw spellings by their normalized form; show groups with more than one spelling."""
    s = df[col].dropna().astype(str)
    groups = s.groupby(s.str.strip().str.lower()).agg(lambda x: x.value_counts().to_dict())
    return groups[groups.apply(len) > 1]

# example: show_variants(hold, "home_ownership")

In [15]:
norm = apps["addr_state"].dropna().astype(str).str.strip().str.lower()
print(norm[~norm.str.fullmatch(r"[a-z]{2}")].value_counts())

addr_state
california        148
texas              93
new york           77
florida            56
georgia            44
pennsylvania       43
new jersey         40
ohio               36
illinois           35
north carolina     32
michigan           24
virginia           24
Name: count, dtype: int64


### Mixed date formats
`issue_d` and `earliest_cr_line` each contain three formats: `Dec-2013` (28,800 rows),
`2013-12-01` (~610) and `12/2013` (~590). Servicing and holdout dates use only the
first. Each format is parsed with its own explicit rule (not pandas' automatic
guessing, which can silently swap day and month), the results are checked for
failures and implausible values, and all dates are stored at month precision.

In [13]:
DATE_FORMATS = {
    r"[A-Za-z]{3}-\d{4}": "%b-%Y",     # Dec-2013
    r"\d{4}-\d{2}-\d{2}": "%Y-%m-%d",  # 2013-12-01
    r"\d{2}/\d{4}": "%m/%Y",           # 12/2013
}

def parse_mixed_dates(s):
    """Parse each known date shape with its own format; anything else becomes NaT."""
    s = s.astype(str).str.strip()
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
    for pattern, fmt in DATE_FORMATS.items():
        mask = s.str.fullmatch(pattern)
        out[mask] = pd.to_datetime(s[mask], format=fmt, errors="coerce")
    return out.dt.to_period("M").dt.to_timestamp()  # month precision

In [14]:
def check_dates(raw, parsed, name):
    """Report unmatched shapes, failed parses, and the parsed range."""
    raw = raw.astype(str).str.strip()
    known = pd.Series(False, index=raw.index)
    for pattern in DATE_FORMATS:
        known |= raw.str.fullmatch(pattern)
    print(f"\n{name}")
    print("  values matching no known format:", int((~known & raw.ne("nan")).sum()))
    print("  matched a format but failed to parse:", int((known & parsed.isna()).sum()))
    print("  range:", parsed.min().date(), "to", parsed.max().date())

    iso = raw[raw.str.fullmatch(r"\d{4}-\d{2}-\d{2}")]
    print("  ISO dates, day part values:", iso.str[-2:].value_counts().to_dict())
    print("  ISO dates, middle part > 12:", int((iso.str[5:7].astype(int) > 12).sum()))

apps["issue_dt"] = parse_mixed_dates(apps["issue_d"])
bureau["earliest_cr_dt"] = parse_mixed_dates(bureau["earliest_cr_line"])
check_dates(apps["issue_d"], apps["issue_dt"], "applications.issue_d")
check_dates(bureau["earliest_cr_line"], bureau["earliest_cr_dt"], "bureau.earliest_cr_line")


applications.issue_d
  values matching no known format: 0
  matched a format but failed to parse: 0
  range: 2013-01-01 to 2015-12-01
  ISO dates, day part values: {'01': 615}
  ISO dates, middle part > 12: 0

bureau.earliest_cr_line
  values matching no known format: 0
  matched a format but failed to parse: 0
  range: 1951-01-01 to 2017-11-01
  ISO dates, day part values: {'01': 607}
  ISO dates, middle part > 12: 0


### Date results and address state
Both date columns parse fully with month precision (no unmatched values, no failed
parses, ISO dates are all day 01, no day/month swaps). However, some `earliest_cr_line`
values fall after the loan was issued, which is impossible and is counted below.
`addr_state` mixes two-letter codes, codes in different cases and full state names;
all are standardized to upper-case two-letter codes.

In [16]:
timeline = apps[["id", "issue_dt"]].merge(bureau[["id", "earliest_cr_dt"]], on="id")
after_issue = timeline["earliest_cr_dt"] > timeline["issue_dt"]
print("earliest_cr_line after issue_d:", int(after_issue.sum()))
display(timeline[after_issue].head())

earliest_cr_line after issue_d: 300


,id,issue_dt,earliest_cr_dt
27,7d1f6edf3c,2013-02-01,2013-11-01
51,13d470c317,2015-05-01,2016-12-01
78,2ba0ab5410,2015-02-01,2017-01-01
86,68732d8749,2015-06-01,2016-09-01
108,8acbab3c06,2013-01-01,2013-04-01


In [17]:
STATE_NAMES = {
    "alabama": "AL", "alaska": "AK", "arizona": "AZ", "arkansas": "AR",
    "california": "CA", "colorado": "CO", "connecticut": "CT", "delaware": "DE",
    "district of columbia": "DC", "florida": "FL", "georgia": "GA", "hawaii": "HI",
    "idaho": "ID", "illinois": "IL", "indiana": "IN", "iowa": "IA", "kansas": "KS",
    "kentucky": "KY", "louisiana": "LA", "maine": "ME", "maryland": "MD",
    "massachusetts": "MA", "michigan": "MI", "minnesota": "MN", "mississippi": "MS",
    "missouri": "MO", "montana": "MT", "nebraska": "NE", "nevada": "NV",
    "new hampshire": "NH", "new jersey": "NJ", "new mexico": "NM", "new york": "NY",
    "north carolina": "NC", "north dakota": "ND", "ohio": "OH", "oklahoma": "OK",
    "oregon": "OR", "pennsylvania": "PA", "rhode island": "RI", "south carolina": "SC",
    "south dakota": "SD", "tennessee": "TN", "texas": "TX", "utah": "UT",
    "vermont": "VT", "virginia": "VA", "washington": "WA", "west virginia": "WV",
    "wisconsin": "WI", "wyoming": "WY",
}
VALID_CODES = set(STATE_NAMES.values())

def clean_state(s):
    """Map full names to codes and upper-case codes; anything unrecognized becomes NaN."""
    norm = s.astype(str).str.strip().str.lower()
    mapped = norm.map(STATE_NAMES).fillna(norm.str.upper())
    return mapped.where(mapped.isin(VALID_CODES))

raw_state = apps["addr_state"].astype(str).str.strip()
apps["addr_state_clean"] = clean_state(apps["addr_state"])

print("written as full name:", int(raw_state.str.lower().isin(STATE_NAMES).sum()))
print("code not in upper case:", int((raw_state.str.fullmatch(r"[A-Za-z]{2}")
                                      & (raw_state != raw_state.str.upper())).sum()))
print("unrecognized after cleaning:", int(apps["addr_state_clean"].isna().sum()))
print("distinct states after cleaning:", apps["addr_state_clean"].nunique())

written as full name: 652
code not in upper case: 1311
unrecognized after cleaning: 0
distinct states after cleaning: 49


In [18]:
log_issue("applications", "issue_d",
          "Mixed date formats: Dec-2013 (28,800), 2013-12-01 (615), 12/2013 (585)",
          rows=1200, action="fix",
          reason="Parsed each format explicitly to month precision; all parse, no day/month swaps")

log_issue("bureau", "earliest_cr_line",
          "Mixed date formats: Dec-2013 (28,800), 2013-12-01 (607), 12/2013 (593)",
          rows=1200, action="fix",
          reason="Same rule as issue_d; all values parse")

log_issue("applications", "addr_state",
          "State written as full name (652) or code not in upper case (1,311)",
          rows=652 + 1311, action="fix",
          reason="Mapped names to two-letter codes and upper-cased; 0 unrecognized. "
                 "Kept for fairness analysis; likely excluded from the model as a location proxy")

### A1.2 summary
- `issue_d` and `earliest_cr_line`: 1,200 rows each in non-standard formats → parsed
  explicitly to month precision, all valid.
- `addr_state`: 1,963 rows in non-standard form (full names or lower case) →
  standardized to two-letter codes.
- Other categories (term, grade, purpose, emp_length, home_ownership,
  verification_status, application_type) have no case or spacing variants.
- Servicing and holdout dates use a single format.
- Noted for A1.4: 300 loans have `earliest_cr_line` after `issue_d` (impossible).